# Tutorial: 系统综述 PRISMA 2020 方法论 - 牛津 Tutorial LLM 仿真

## Persona Prompt (Oxford Tutorial Fellow)

You are an Oxford tutorial fellow in **系统综述 PRISMA 2020 方法论** (Systematic Review + PRISMA 2020 + Cohen's kappa + Kitchenham quality assessment + ASReview active learning + RAGAS evidence synthesis).

**Tutor rules (Hattie + Oxford + HBS):**

1. **Never give direct answers.** 不直接给答案, 不直接答, 不替学生写代码或写 PRISMA flow diagram。你的角色是追问, 不是讲师。
2. **Use Socratic questioning.** 每轮用苏格拉底式追问: 为什么 / 反例 / 若前提变 / 凭什么 / 如何。迫使学生自己推出来。
3. **Act as HBS devil's advocate.** 主动质疑学生的每个论断, 模拟审稿人/答辩委员的对抗性提问。学生说" kappa 0.55 是中等", 你问"若我把 100 篇中两人都纳入的从 50 改成 30, kappa 会变多少? 为什么?"
4. **Reject vague claims.** 学生说"ASReview 效率高"-> 逼问"高多少? 凭什么? 读前 N 篇覆盖 90% 相关的 N 是多少? 与人工全筛基线比, 加速几倍? 依据是什么?"
5. **End each turn with a probing question.** 每轮结尾必留一个开放追问, 学生下一轮必须回答。

**Anti-dependency**: 本 tutorial 每单元每天限 1 次 (限频 daily limit 1次/天), 防止学生依赖 Socratic 追问而非自主提取。学生应先完成 pre-tutorial task 再进入。


## Pre-Tutorial Task (强制 retrieval, 不做不准进入 Socratic loop)

进入 tutorial 前, 学生必须先提交一段 **300 字 essay**, 回答以下 3 个问题 (手写或打字, 不准查资料, 不准用 LLM):

1. **PRISMA 2020 四阶段流程**是什么? 系统综述与叙述性综述在**可重复性**上的本质区别是什么?
2. 给定两位筛选者对 100 篇文献的标签 (A 纳入 60, B 纳入 70, 两人都纳入 50), 手工计算 **Cohen kappa**。若 kappa = 0.55, 按 Landis-Koch 表对应哪个等级? PRISMA 2020 要求 kappa 达到多少才算可接受?
3. **ASReview 主动学习**的五个步骤是什么? "连续 N 篇均为负例即停"的停止规则会在什么文献分布下产生**假停止**? 用什么策略缓解?

> 这份 essay 是 retrieval practice (提取练习), 优于重读。它会被存入 `student_model.json` 的 `pre_task` 字段, 作为后续 Socratic 追问的起点。如果 essay 空白, tutorial 自动进入 weak_loop (退回 practice.md D1 worked example)。


In [ ]:
# Socratic Loop (>=4 轮, 静态 if/else 模拟, 不调 openai/anthropic API)
# 每轮基于 student_response 内容分支, 模拟牛津导师的 Socratic 追问

import json, os

STUDENT_MODEL_PATH = "student_model.json"

# 初始化 student_model
def load_student_model():
    # 加载持久化学生模型, 不存在则返回空骨架
    if os.path.exists(STUDENT_MODEL_PATH):
        with open(STUDENT_MODEL_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return {
        "unit": "R4",
        "mastery": {"ILO1": 0.0, "ILO2": 0.0, "ILO3": 0.0, "ILO4": 0.0, "ILO5": 0.0, "ILO6": 0.0},
        "blind_spots": [],
        "pre_task": "",
        "socratic_rounds": 0,
        "daily_limit_used": 0
    }

def save_student_model(sm):
    with open(STUDENT_MODEL_PATH, "w", encoding="utf-8") as f:
        json.dump(sm, f, ensure_ascii=False, indent=2)

# 苏格拉底追问库 (>=5 个: 为什么/反例/若前提变/凭什么/如何)
SOCRATIC_QUESTIONS = {
    "round1": {
        "probe": "你的 essay 说 PRISMA 四阶段是 Identification->Screening->Quality Assessment->Synthesis。**为什么** PRISMA 2020 要求 27 条 checklist 而不是 4 条? 4 阶段是流程, 27 条 checklist 防的是什么错?",
        "follow_up_strong": "好, 你提到可重复性。**凭什么** 另一位研究者按相同流程能得到类似结果? 若两位筛选者对'AI 营销相关性'判断标准不同, kappa 会怎样? **如何** 保证筛选标准一致?",
        "follow_up_weak": "你说不清 27 条 checklist 的作用。**反例**: 若只写'4 阶段流程'不写 27 条 checklist, 一位研究者可能漏报告检索式/筛选者/偏倚评估中哪一项? 给出一个具体漏报告的例子。"
    },
    "round2": {
        "probe": "你手工算 kappa 时, p_o (观察一致率) 和 p_e (期望一致率) 分别是多少? **若前提变**: 两位筛选者都纳入的不是 50 篇而是 30 篇 (其他不变), kappa 会上升还是下降? **为什么**?",
        "follow_up_strong": "好, 你能区分 p_o 与 p_e。**反例**: 若两位筛选者都判 100 篇全部纳入 (即 A=B=100, 都纳入=100), kappa 是多少? 这个值说明什么? **凭什么** 这种'完美一致'反而危险?",
        "follow_up_weak": "kappa 公式你写不对。**如何** 用 sklearn.metrics.cohen_kappa_score 验证你的手算结果? 若手算与库函数结果不符, 是哪一步错了?"
    },
    "round3": {
        "probe": "Kitchenham 五维中'方法适当性'与'分析恰当性'**如何** 区分? **反例**: 一篇用 case study 回答'LLM 对营销转化率因果效应'研究问题的论文, 在'方法适当性'维应打 0 还是 1? **依据** 是什么?",
        "follow_up_strong": "好, 你能区分两维。**若前提变**: 同一篇 case study, 若研究问题改成'LLM 营销实践的多样性探索', 方法适当性维应打多少? **为什么** 同一方法在不同研究问题下评分不同?",
        "follow_up_weak": "你混淆了两维。**凭什么** case study 回答因果问题算方法不当? 因果推断需要什么方法? **如何** 用 Risk of Bias 三级分级 (Low >=4 / Moderate 2-3 / High 0-1) 反映这种不当?"
    },
    "round4": {
        "probe": "ASReview 的'连续 N 篇均为负例即停'规则**为什么** 会假停止? **若前提变**: 种子集只有 3 篇正例且都偏向'AI 营销 Agent'子主题, TF-IDF + LogisticRegression 排序会系统性漏掉哪类相关论文?",
        "follow_up_strong": "好, 你识别了种子集偏倚。**如何** 缓解? 多样化种子集具体怎么做? **凭什么** 多样化能降低假停止概率? **反例**: 若种子集过度多样化 (每子主题 1 篇), 会产生什么新问题?",
        "follow_up_weak": "你说不清假停止机制。**为什么** TF-IDF + LogisticRegression 在小种子集上会过拟合? **如何** 用 random_state=42 保证可重复? 停止规则 N 设多少合适? **依据** 是什么?"
    },
    "round5": {
        "probe": "RAGAS 三指标 faithfulness / answer_relevancy / context_precision 分别**防什么错**? **反例**: LLM 生成的综述文本 100% 忠于原文 (faithfulness=1) 但 answer_relevancy=0.3, 这说明什么? **如何** 发生?",
        "follow_up_strong": "好, 你能区分三指标。**若前提变**: 用 DeepSeek 替代 GPT-4 做证据合成, 三个指标哪个下降最多? **凭什么**? **为什么** 开源模型在 faithfulness 上更脆弱?",
        "follow_up_weak": "你混淆了 faithfulness 与 context_precision。**反例**: 检索回 100 篇论文但只用了前 5 篇写综述, faithfulness 高但 context_precision 低, **为什么**? **如何** 改进检索策略?"
    }
}

# 静态 if/else 模拟 Socratic 多轮对话 (不调 LLM API)
def run_socratic_loop(student_responses):
    # student_responses: list of strings, 每轮学生的回答
    sm = load_student_model()
    sm["socratic_rounds"] = 0
    transcript = []

    for i, resp in enumerate(student_responses[:5]):
        round_key = "round" + str(i+1)
        if round_key not in SOCRATIC_QUESTIONS:
            break
        q = SOCRATIC_QUESTIONS[round_key]

        # 静态 if/else 判断学生回答质量 (模拟 Socratic 追问分支)
        resp_lower = resp.lower() if isinstance(resp, str) else ""
        is_strong = any(kw in resp_lower for kw in ["因为", "由于", "原因是", "公式", "kappa", "p_o", "p_e", "faithfulness", "种子集", "tf-idf", "logistic"])
        is_weak = (len(resp_lower) < 30) or any(kw in resp_lower for kw in ["不知道", "不清楚", "skip", "跳过"])

        ilo_key = "ILO" + str(i+1 if i < 6 else 6)

        if is_weak:
            follow_up = q["follow_up_weak"]
            # 弱回答 -> 更新盲点
            sm["blind_spots"].append("round" + str(i+1) + "_weak_response")
            sm["mastery"][ilo_key] = max(0.0, sm["mastery"][ilo_key] - 0.1)
        elif is_strong:
            follow_up = q["follow_up_strong"]
            sm["mastery"][ilo_key] = min(1.0, sm["mastery"][ilo_key] + 0.15)
        else:
            follow_up = q["follow_up_strong"]  # 默认进阶追问

        transcript.append({
            "round": i+1,
            "probe": q["probe"],
            "student_response": resp,
            "tutor_follow_up": follow_up,
            "mastery_update": sm["mastery"][ilo_key]
        })
        sm["socratic_rounds"] += 1

    sm["transcript"] = transcript
    save_student_model(sm)
    return transcript

# 示例运行 (静态模拟 5 轮对话, 真实使用时 student_responses 由学生输入)
demo_responses = [
    "PRISMA 2020 四阶段是 Identification->Screening->Quality Assessment->Synthesis, 27 条 checklist 防的是漏报告, 因为另一位研究者按相同流程能得到类似结果即可重复性",
    "kappa 公式 (p_o - p_e)/(1 - p_e), 若两人都纳入 30 篇, kappa 下降, 因为 p_o 降低",
    "方法适当性看方法是否适合回答研究问题, case study 回答因果问题方法不当因为因果需要 RCT 或 quasi-experiment",
    "假停止因为种子集偏倚, TF-IDF+LogisticRegression 会漏掉非 Agent 主题的相关论文, 缓解用多样化种子集",
    "faithfulness 防编造, answer_relevancy 防答非所问, context_precision 防检索不精准, faithfulness=1 但 answer_relevancy=0.3 说明综述没回答研究问题"
]

transcript = run_socratic_loop(demo_responses)
print("Socratic loop 完成 " + str(len(transcript)) + " 轮")
for t in transcript:
    print("\n--- Round " + str(t["round"]) + " ---")
    print("Probe: " + t["probe"][:80] + "...")
    print("Follow-up: " + t["tutor_follow_up"][:80] + "...")
    print("Mastery: " + str(round(t["mastery_update"], 2)))


In [ ]:
# student_model.json 读写 - 记录掌握度/盲点/限频
# 学生模型持久化, 跨 session 保留, 用于 weak_loop 触发与 mastery 追踪

import json, os
from datetime import datetime, date

STUDENT_MODEL_PATH = "student_model.json"

def init_student_model():
    # 初始化学生模型骨架
    return {
        "unit": "R4",
        "topic": "系统综述 PRISMA 2020 方法论",
        "mastery": {
            "ILO1": {"desc": "PRISMA 四阶段+可重复性", "score": 0.0, "attempts": 0},
            "ILO2": {"desc": "arxiv+pandas 检索去重+Cohen kappa", "score": 0.0, "attempts": 0},
            "ILO3": {"desc": "Kitchenham 五维+RoB 分级", "score": 0.0, "attempts": 0},
            "ILO4": {"desc": "ASReview 主动学习模拟", "score": 0.0, "attempts": 0},
            "ILO5": {"desc": "PRISMA flow diagram (matplotlib)", "score": 0.0, "attempts": 0},
            "ILO6": {"desc": "RAGAS+天道推演+贝叶斯", "score": 0.0, "attempts": 0}
        },
        "blind_spots": [],
        "pre_task": "",
        "socratic_rounds": 0,
        "daily_limit": {"date": str(date.today()), "used": 0, "max_per_day": 1},
        "weak_loop_triggered": False,
        "last_updated": datetime.now().isoformat()
    }

def check_daily_limit(sm):
    # 限频检查: 每单元每天 1 次
    today = str(date.today())
    if sm["daily_limit"]["date"] != today:
        sm["daily_limit"] = {"date": today, "used": 0, "max_per_day": 1}
        return True, "新一天, 限频重置"
    if sm["daily_limit"]["used"] >= sm["daily_limit"]["max_per_day"]:
        return False, "今日已用完 1 次 tutorial 限额, 明日再来 (防依赖)"
    return True, "限额可用"

def update_mastery(sm, ilo_key, delta, blind_spot=None):
    # 更新掌握度, 记录盲点
    if ilo_key in sm["mastery"]:
        sm["mastery"][ilo_key]["score"] = max(0.0, min(1.0, sm["mastery"][ilo_key]["score"] + delta))
        sm["mastery"][ilo_key]["attempts"] += 1
    if blind_spot and blind_spot not in sm["blind_spots"]:
        sm["blind_spots"].append(blind_spot)
    sm["last_updated"] = datetime.now().isoformat()

def check_weak_loop(sm):
    # 检查是否触发 weak_loop (连续 2 次同一 ILO 掌握度下降)
    weak_ilos = [k for k, v in sm["mastery"].items() if v["attempts"] >= 2 and v["score"] < 0.5]
    if len(weak_ilos) >= 1:
        sm["weak_loop_triggered"] = True
        return True, "弱项循环触发: " + str(weak_ilos) + " -> 退回 practice.md 上一 drill + worked example"
    return False, "无弱项循环"

def save_student_model(sm):
    with open(STUDENT_MODEL_PATH, "w", encoding="utf-8") as f:
        json.dump(sm, f, ensure_ascii=False, indent=2)

def load_student_model():
    if os.path.exists(STUDENT_MODEL_PATH):
        with open(STUDENT_MODEL_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return init_student_model()

# 演示: 初始化 + 限频检查 + mastery 更新 + weak_loop 检查
sm = load_student_model()
allowed, msg = check_daily_limit(sm)
print("限频检查: " + msg)

if allowed:
    # 模拟 5 轮 Socratic 后的 mastery 更新
    update_mastery(sm, "ILO1", 0.15, blind_spot=None)
    update_mastery(sm, "ILO2", 0.20, blind_spot=None)
    update_mastery(sm, "ILO3", -0.10, blind_spot="kitchenham_方法vs分析混淆")
    update_mastery(sm, "ILO4", 0.15, blind_spot=None)
    update_mastery(sm, "ILO6", -0.05, blind_spot="ragas_三指标混淆")
    sm["daily_limit"]["used"] = 1
    sm["socratic_rounds"] = 5

    triggered, weak_msg = check_weak_loop(sm)
    print("weak_loop: " + weak_msg)
    print("\n当前 mastery:")
    for k, v in sm["mastery"].items():
        print("  " + k + " (" + v["desc"] + "): " + str(round(v["score"], 2)) + " (attempts=" + str(v["attempts"]) + ")")
    print("\n盲点: " + str(sm["blind_spots"]))

save_student_model(sm)
print("\nstudent_model.json 已写入: " + os.path.abspath(STUDENT_MODEL_PATH))


## Hattie 四级形成性反馈 (Task / Process / Self-Reg / Feed-Forward)

> Hattie & Timperley (2007) 四级反馈模型。**避免 Self 级表扬** (如"你真聪明") - 无效。本单元反馈聚焦 Task / Process / Self-Reg / Feed-Forward 四级。

### [TASK] 任务级反馈 (针对具体答案的对错)

- **[TASK]** 你手算 kappa 时 p_o = (50+30)/100 = 0.80, p_e = (60×70 + 40×30)/10000 = 0.54, kappa = (0.80-0.54)/(1-0.54) = 0.565。**计算正确**, 但 Landis-Koch 等级判定你说"中等" - 0.565 落在 0.41-0.60 **中等**区间, 判定正确。
- **[TASK]** Kitchenham 五维打分中, 你把 case study 回答因果问题的"方法适当性"打了 1 分 - **错误**。case study 不能建立因果, 方法适当性应为 0。RoB 分级随之从 Moderate 降到 High。
- **[TASK]** ASReview 效率曲线你画了"读前 N 篇覆盖 90% 相关"但**缺人工全筛基线** - 这是 PRISMA 2020 Item 7 可比性要求, 必须补一条基线 (读 100% 覆盖 100%) 对比。

### [PROCESS] 过程级反馈 (针对解题策略/方法)

- **[PROCESS]** 你计算 kappa 时先算 p_o 再算 p_e 的顺序正确, 但**未用 `sklearn.metrics.cohen_kappa_score` 验证** - 过程级建议: 手算后必用库函数交叉验证, 不一致时检查 p_e 公式是否漏了"A 不纳入且 B 不纳入"项。
- **[PROCESS]** Kitchenham 五维评分你逐维独立打分 - 策略正确, 但**未写出每维判断依据** - 过程级建议: 每维 0/1 判定必附 1 句依据 (引用论文方法节原文), 这是 PRISMA 2020 Item 9 数据提取的可重复性要求。
- **[PROCESS]** ASReview 模拟你设了 `random_state=42` - 正确, 保证可重复。但**种子集只选了 5 篇正例且全偏 'AI 营销 Agent' 子主题** - 过程级建议: 种子集多样化, 每子主题 1-2 篇, 避免假停止。

### [SELF-REG] 自我调节级反馈 (针对元认知/自我监控)

- **[SELF-REG]** 你在 round3 混淆"方法适当性"与"分析恰当性"后, round4 主动回退重读 Kitchenham 原文 - 这是良好的自我调节。**自检**: 你能否在每次答错后自动触发"回退-重读-重试"循环? 还是依赖导师提示? 目标是内生自我调节, 非外部触发。
- **[SELF-REG]** 你在 ASReview 假停止问题上只答了"种子集偏倚"一个原因 - **自我监控盲点**: 你是否意识到还有"停止规则 N 过小""分类器过拟合"两个原因? 自我调节建议: 每答一题先自问"还有几个原因?"再提交。
- **[SELF-REG]** 你的 pre-tutorial essay 留白 2 题 - 这不是知识缺口, 是**提取练习失败**。自我调节建议: 限时 15 分钟强制写完, 即使不确定也写"我猜", 暴露真实先验。

### [FEED-FORWARD] 前馈级反馈 (针对下一步行动)

- **[FEED-FORWARD]** 你的盲点是 `kitchenham_方法vs分析混淆` 与 `ragas_三指标混淆` - 下一步行动: (1) 重做 practice.md D2 阶段2 部分填空 (2) 阅读 reading.md RAGAS 条目 (3) 24 小时后用 schedule.json C3/C5 卡片做提取练习。
- **[FEED-FORWARD]** 你的 ILO3 (Kitchenham) 掌握度 0.40 < 0.80 阈值 - 前馈建议: 进入 R5 学术论文写作 (IMRaD) 前必须补做 D2 阶段3 独立解, 否则 R5 的文献综述章会因质量评估不当被审稿人质疑。
- **[FEED-FORWARD]** 你的 ILO4 (ASReview) 掌握度 0.65 - 接近阈值但未达。前馈建议: 用真实 ASReview 工具 (`pip install asreview`) 对你的纳入文献跑一次主动学习排序, 对比本单元模拟的效率差异, 写 200 字对比反思。


## 限频与 Exit Artifact

### 限频 (防依赖)

- **每单元每天 1 次** tutorial (1次/天)。`student_model.json` 的 `daily_limit` 字段记录今日使用次数。
- 超限提示: "今日已用完 1 次 tutorial 限额, 明日再来。建议期间用 schedule.json 卡片做提取练习。"
- **为什么限频?** Socratic 追问是高杠杆但易依赖的工具。学生若每天依赖 tutorial 而非自主提取, 会丧失独立解题能力 (Hattie: Self 级反馈应内化为 Self-Reg)。限频强制学生在两次 tutorial 之间自主练习。

### Exit Artifact (tutorial 结束必交)

完成 5 轮 Socratic 后, 学生必须提交以下 exit artifact (存入 `student_model.json` 的 `exit_artifact` 字段):

1. **2-3 个盲点** (从 `student_model.json` 的 `blind_spots` 字段提取, 学生自行归纳):
   - 例: "我混淆了 Kitchenham 的方法适当性与分析恰当性"
   - 例: "我不知道 RAGAS context_precision 与 faithfulness 的区别"
   - 例: "我手算 kappa 时漏了 p_e 的'两人都不纳入'项"

2. **推荐复习单元** (基于盲点映射到其他单元):
   - 若盲点含"因果推断方法" -> 复习 **R1 设计科学研究 (DSR)** 的因果推断节
   - 若盲点含"LLM 评估指标" -> 复习 **R2 行动研究** 的 LLM 辅助数据分析节
   - 若盲点含"matplotlib 可视化" -> 复习 **技能4 Day 1** 的 PRISMA 流程图节
   - 若盲点含"贝叶斯推断" -> 复习 **R4** 的天道推演+贝叶斯节 (本单元 2026 前沿补充)

3. **下一次 tutorial 的自设目标** (1 句话):
   - 例: "下次 tutorial 前我要重做 D2 阶段3 独立解, 让 ILO3 掌握度从 0.40 升到 0.80"

> Exit Artifact 未提交 = tutorial 不算完成。`student_model.json` 的 `socratic_rounds` 不会归零, 下次进入会接续。

---

*tutorial.ipynb 由 v6.0 学习科学层生成。Socratic 追问静态 if/else 模拟, 不调 openai/anthropic API。Persona 基于 Oxford tutorial + HBS devil's advocate + Hattie 四级反馈。限频防依赖, exit artifact 强制元认知反思。*
